<center><h1> WR if only nth result counts </h1></center>

What would the world record be of only the nth best result from each competitor counted for the rankings?

### Imports

In [1]:
import pandas as pd
import numpy as np

In [2]:
Results = pd.read_csv('WCA_export_Results.tsv', sep='\t')

In [3]:
Competitions = pd.read_csv('WCA_export_Competitions.tsv', sep='\t')

In [4]:
df = Results.merge(Competitions, how='left', left_on='competitionId', right_on='id', validate = "m:1")
df = df.drop('id', 1)

In [5]:
df = df.rename(columns = {'name':'competitionName'})

In [6]:
rounds = pd.read_csv('WCA_export_RoundTypes.tsv', sep='\t', low_memory = False)

In [7]:
df = df.merge(rounds[['id','rank']], how='left', left_on='roundTypeId', right_on='id')
df = df.drop('id',1)

The final dataset looks like this:

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 2870882 entries, 0 to 2870881
Data columns (total 38 columns):
 #   Column                 Dtype 
---  ------                 ----- 
 0   competitionId          object
 1   eventId                object
 2   roundTypeId            object
 3   pos                    int64 
 4   best                   int64 
 5   average                int64 
 6   personName             object
 7   personId               object
 8   personCountryId        object
 9   formatId               object
 10  value1                 int64 
 11  value2                 int64 
 12  value3                 int64 
 13  value4                 int64 
 14  value5                 int64 
 15  regionalSingleRecord   object
 16  regionalAverageRecord  object
 17  competitionName        object
 18  cityName               object
 19  countryId              object
 20  information            object
 21  year                   int64 
 22  month                  int64 
 23  day    

Other imports

In [9]:
ss = pd.read_csv('WCA_export_RanksSingle.tsv', sep='\t', low_memory = False)

In [10]:
aa = pd.read_csv('WCA_export_RanksSingle.tsv', sep='\t', low_memory = False)

In [11]:
pp = pd.read_csv('WCA_export_Persons.tsv', sep='\t', low_memory = False)

event list

In [12]:
eventi = list(set(df[df['year']>2020]['eventId'])) #take current events
eventi

['222',
 '444',
 '555bf',
 '333bf',
 'minx',
 '333',
 '777',
 '666',
 'pyram',
 '333fm',
 'clock',
 '555',
 'skewb',
 '444bf',
 '333oh',
 'sq1',
 '333mbf']

# WRs counting nth best result

Reduce size to make code faster. it's likely that "WR if only nth best result counts" is in top50 single+top50avg

In [13]:
ss = ss[ss['worldRank'] <= 50]

In [14]:
aa = aa[aa['worldRank'] <= 50]

In [15]:
persone = list(set(list(set(aa['personId']))+list(set(aa['personId']))))

## Singles

In [16]:
n = 2 #choose n
n = n-1 #because 0 = 1

In [17]:
dict_s = {}

persone = list(set(ss['personId']))

df1 = df[['personId','eventId','value1','value2','value3','value4','value5']]

for p in persone:
    subset = df1[df1['personId'] == p]
    l = []
    for e in eventi:
        solve = subset[subset['eventId'] == e]
        solve_l = list(solve[solve['value1']>0]['value1']) + list(solve[solve['value2']>0]['value2']) + list(solve[solve['value3']>0]['value3']) + list(solve[solve['value4']>0]['value4']) + list(solve[solve['value5']>0]['value5'])
        if len(solve_l) < n: #less than N results for event
            l.append((e,np.nan))
            continue
        elif len(solve_l) == n: #n results for event
            l.append((e,max(solve_l)))
            continue
        else:
            a = np.partition(solve_l,n)[n] #more than N results. Partition only puts nth in nth position
            l.append((e,a))
    
    dict_s[p] = l

Modeling the problem as "for each person, for each event, take solves, get nth if exists", I think the only improvement in efficieny could be in the way I get all solves into a list (using .melt maybe).
Maybe filtering for events only is faster

### df for singles

In [18]:
singles = pd.DataFrame(([k, elt[0], elt[1]] for  k,v in dict_s.items() for elt in v),
             columns = ['id', 'event', 'time'])

In [19]:
singles['rank'] = singles.groupby('event').rank(method = 'min', axis = 1) #rankings per event

In [20]:
singles = singles[singles['rank'] == 1] #only WR

In [21]:
#cleaning
singles = singles.dropna()
singles = singles.merge(pp[['id','name']], on = 'id').reset_index(drop = True)
singles = singles[['id','name','event','time']]
singles = singles.sort_values(by = 'event')

In [22]:
singles

,id,name,event,time
15,2014ZYCH01,Jan Zych,222,61.0
7,2012PARK03,Max Park,333,418.0
12,2015CHER07,Tommy Cherry,333bf,1467.0
5,2011TRON02,Sebastiano Tronto,333fm,18.0
14,2016SIGG01,Graham Siggins,333mbf,430360002.0
11,2012PARK03,Max Park,333oh,699.0
6,2012PARK03,Max Park,444,1686.0
2,2016CHAP04,Stanley Chapel,444bf,6251.0
10,2012PARK03,Max Park,555,3492.0
1,2016CHAP04,Stanley Chapel,555bf,14880.0


## Averages

In [23]:
dict_a = {}

df2 = df[['personId','eventId','average']]

for p in persone:
    subset = df2[df2['personId'] == p]
    l = []
    for e in eventi:
        solve = subset[subset['eventId'] == e]
        solve_l = list(solve[solve['average']>0]['average'])
        if len(solve_l) < n:
            l.append((e,np.nan))
            continue
        elif len(solve_l) == n:
            l.append((e,max(solve_l)))
            continue
        else:
            a = np.partition(solve_l,n)[n]
            l.append((e,a))
    
    dict_a[p] = l

### df for singles

In [24]:
avgs = pd.DataFrame(([k, elt[0], elt[1]] for  k,v in dict_a.items() for elt in v),
             columns = ['id', 'event', 'time'])

In [25]:
avgs['rank'] = avgs.groupby('event').rank(method = 'min', axis = 1)

In [26]:
avgs = avgs[avgs['rank'] == 1]

In [27]:
avgs = avgs.dropna()
avgs = avgs.merge(pp[['id','name']], on = 'id').reset_index(drop = True)
avgs = avgs[['id','name','event','time']]
avgs = avgs.sort_values(by = 'event')

In [28]:
avgs

,id,name,event,time
6,2018KHAN28,Zayn Khanani,222,103.0
15,2016KOLA02,Tymon Kolasiński,333,515.0
13,2015CHER07,Tommy Cherry,333bf,1750.0
2,2018LOVE03,Jack Love,333fm,2233.0
5,2014SCHO02,Cale Schoon,333fm,2233.0
12,2012PONC02,Patrick Ponce,333oh,870.0
8,2012PARK03,Max Park,444,2014.0
3,2016CHAP04,Stanley Chapel,444bf,7255.0
11,2012PARK03,Max Park,555,3861.0
7,2013LINK01,Kaijun Lin (林恺俊),555bf,18321.0
